# 🎯 KAGGLE GPU EVALUATION: BASE MODEL TRÊN 15 LOẠI HÓA ĐƠN TIẾNG VIỆT
Đánh giá định lượng thực tế 100% trên GPU NVIDIA Tesla T4 (16GB VRAM) của Base Model Qwen2-VL-2B.

In [ ]:
# 1. Cài đặt môi trường chuẩn xác
!pip uninstall -y -q torchao
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers==4.46.2" "peft==0.13.2" "accelerate==0.34.2" pillow torchvision

import sys
for mod in list(sys.modules.keys()):
    if any(mod.startswith(k) for k in ["transformers", "peft", "accelerate", "torchao", "qwen_vl_utils"]):
        del sys.modules[mod]

import os
import time
import json
import re
import zipfile
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
print(f"🧠 Tổng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Hàm tính metrics ANLS và Exact Match chuẩn xác
def levenshtein_distance(s1: str, s2: str) -> int:
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def calculate_anls(prediction: str, ground_truth: str, threshold: float = 0.5) -> float:
    p = str(prediction).strip().lower()
    gt = str(ground_truth).strip().lower()
    if not p and not gt:
        return 1.0
    if not p or not gt:
        return 0.0
    dist = levenshtein_distance(p, gt)
    max_len = max(len(p), len(gt))
    norm_dist = dist / max_len
    if norm_dist < threshold:
        return 1.0 - norm_dist
    return 0.0

def calculate_exact_match(prediction: str, ground_truth: str) -> float:
    return 1.0 if str(prediction).strip().lower() == str(ground_truth).strip().lower() else 0.0


In [ ]:
# 3. Giải nén và lập chỉ mục 15 loại ảnh hóa đơn
extract_dir = "/kaggle/working/extracted_images"
os.makedirs(extract_dir, exist_ok=True)

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "images.zip":
            print(f"📦 Đang giải nén {f}...")
            with zipfile.ZipFile(os.path.join(root, f), 'r') as zf:
                zf.extractall(extract_dir)

image_map = {}
for root, dirs, files in os.walk("/kaggle"):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            image_map[f] = os.path.join(root, f)

print(f"📸 Đã lập chỉ mục {len(image_map)} ảnh hóa đơn trong hệ thống!")

# Danh sách câu hỏi kiểm định thực tế của 15 loại hóa đơn
validation_samples = [
  {
    "id": 1,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY CỔ PHẦN ĐẦU TƯ & PHÁT TRIỂN HƯNG PHÁT"
  },
  {
    "id": 2,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "24,389,200đ"
  },
  {
    "id": 3,
    "template": "einvoice_viettel",
    "image_name": "einvoice_viettel_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "13/06/2026 15:43"
  },
  {
    "id": 9,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CÔNG TY TNHH PHÁT TRIỂN CÔNG NGHỆ HOÀNG PHÁT"
  },
  {
    "id": 10,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "7,326,000đ"
  },
  {
    "id": 11,
    "template": "einvoice_vnpt",
    "image_name": "einvoice_vnpt_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "06/06/2026 11:59"
  },
  {
    "id": 17,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "TRUNG TÂM Y TẾ HUYỆN GIA LÂM"
  },
  {
    "id": 18,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "165.000 VND"
  },
  {
    "id": 19,
    "template": "receipt_c45_bb",
    "image_name": "receipt_c45_bb_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "Ngày 17 tháng 06 năm 2026"
  },
  {
    "id": 25,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "VINCOMMERCE"
  },
  {
    "id": 26,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "94.050"
  },
  {
    "id": 27,
    "template": "supermarket_winmart",
    "image_name": "supermarket_winmart_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "12/06/2026 13:46"
  },
  {
    "id": 33,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "SAIGON CO.OP"
  },
  {
    "id": 34,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "759,550.00"
  },
  {
    "id": 35,
    "template": "supermarket_lotte",
    "image_name": "supermarket_lotte_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 18:55:00"
  },
  {
    "id": 41,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "BÁCH HÓA XANH"
  },
  {
    "id": 42,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "233,200"
  },
  {
    "id": 43,
    "template": "supermarket_bachhoaxanh",
    "image_name": "supermarket_bachhoaxanh_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "21/05/2026 13:25"
  },
  {
    "id": 49,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CIRCLE K"
  },
  {
    "id": 50,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "92,400đ"
  },
  {
    "id": 51,
    "template": "convenience_circlek",
    "image_name": "convenience_circlek_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "07/06/2026 14:03"
  },
  {
    "id": 57,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "GS25 TÔN THẤT THUYẾT"
  },
  {
    "id": 58,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "117,720đ"
  },
  {
    "id": 59,
    "template": "convenience_gs25",
    "image_name": "convenience_gs25_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "30/05/2026 16:59"
  },
  {
    "id": 65,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "7-Eleven Saigon Trade Center"
  },
  {
    "id": 66,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "66,000"
  },
  {
    "id": 67,
    "template": "convenience_7eleven",
    "image_name": "convenience_7eleven_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "29/05/2026 22:37"
  },
  {
    "id": 71,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "HIGHLANDS COFFEE TRẦN HƯNG ĐẠO"
  },
  {
    "id": 72,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "796,068"
  },
  {
    "id": 73,
    "template": "cafe_highlands",
    "image_name": "cafe_highlands_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "31/05/2026 16:41"
  },
  {
    "id": 79,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "PHÚC LONG NGUYỄN VĂN CỪ"
  },
  {
    "id": 80,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "506,825"
  },
  {
    "id": 81,
    "template": "cafe_phuclong",
    "image_name": "cafe_phuclong_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "01/06/2026 21:19"
  },
  {
    "id": 87,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "STARBUCKS HÀN THUYÊN"
  },
  {
    "id": 88,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "841,500"
  },
  {
    "id": 89,
    "template": "cafe_starbucks",
    "image_name": "cafe_starbucks_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "03/06/2026"
  },
  {
    "id": 95,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "KFC Bà Hom"
  },
  {
    "id": 96,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "437,800"
  },
  {
    "id": 97,
    "template": "restaurant_kfc",
    "image_name": "restaurant_kfc_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 15:04"
  },
  {
    "id": 103,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "JOLLIBEE COOPMART"
  },
  {
    "id": 104,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "127,440đ"
  },
  {
    "id": 105,
    "template": "restaurant_jollibee",
    "image_name": "restaurant_jollibee_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 21:31"
  },
  {
    "id": 111,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "SELLER",
    "question": "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
    "ground_truth": "CỬA HÀNG TIỆN LỢI AN AN"
  },
  {
    "id": 112,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "TOTAL_COST",
    "question": "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
    "ground_truth": "207.900"
  },
  {
    "id": 113,
    "template": "minimart_anan",
    "image_name": "minimart_anan_val_001.png",
    "field": "TIMESTAMP",
    "question": "Ngày giờ lập hóa đơn là khi nào?",
    "ground_truth": "02/06/2026 20:13"
  }
]

# Khớp ảnh thật với câu hỏi
matched_samples = []
for s in validation_samples:
    img_name = s["image_name"]
    if img_name in image_map:
        s["full_image_path"] = image_map[img_name]
        matched_samples.append(s)

print(f"🎯 Khớp thành công {len(matched_samples)} / {len(validation_samples)} mẫu kiểm thử có ảnh thật!")


In [ ]:
# 4. Nạp Base Model Qwen2-VL-2B-Instruct vào VRAM
model_name = "Qwen/Qwen2-VL-2B-Instruct"
print(f"⏳ Đang nạp Base Model: {model_name} (FP16)... ")
processor = AutoProcessor.from_pretrained(model_name, min_pixels=256*28*28, max_pixels=1024*28*28)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("✅ Nạp thành công Base Model vào GPU!")


In [ ]:
# 5. Thực thi suy luận THỰC TẾ và đo đạc chỉ số
print("=" * 75)
print("🚀 BẮT ĐẦU CHẠY SUY LUẬN BASE MODEL THỰC TẾ TRÊN 15 LOẠI HÓA ĐƠN...")
print("=" * 75)

results = []
total_anls = 0.0
total_em = 0.0
latencies = []
template_stats = {}

for idx, sample in enumerate(matched_samples):
    img_path = sample["full_image_path"]
    question = sample["question"]
    gt = sample["ground_truth"]
    tmpl = sample.get("template", "unknown")
    
    if tmpl not in template_stats:
        template_stats[tmpl] = {"count": 0, "anls": 0.0, "em": 0.0}
    
    t0 = time.time()
    image = Image.open(img_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        prediction = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()
        
    latency = time.time() - t0
    latencies.append(latency)
    
    anls_score = calculate_anls(prediction, gt)
    em_score = calculate_exact_match(prediction, gt)
    
    total_anls += anls_score
    total_em += em_score
    
    template_stats[tmpl]["count"] += 1
    template_stats[tmpl]["anls"] += anls_score
    template_stats[tmpl]["em"] += em_score
    
    results.append({
        "id": idx + 1,
        "template": tmpl,
        "image": sample["image_name"],
        "question": question,
        "ground_truth": gt,
        "prediction": prediction,
        "anls": round(anls_score, 4),
        "exact_match": int(em_score),
        "latency_seconds": round(latency, 3)
    })
    
    print(f"[{idx+1}/{len(matched_samples)}] ({tmpl}) {sample['image_name']} | Latency: {latency:.2f}s | ANLS: {anls_score:.2f} | EM: {int(em_score)}")
    print(f"   ❓ Q:  {question}")
    print(f"   🎯 GT: {gt}")
    print(f"   🤖 PR: {prediction}")
    print("-" * 75)

num_tests = len(matched_samples)
avg_anls = total_anls / num_tests if num_tests > 0 else 0.0
avg_em = total_em / num_tests if num_tests > 0 else 0.0
avg_lat = sum(latencies) / len(latencies) if latencies else 0.0

# Thống kê từng loại hóa đơn
template_breakdown = []
for t, d in template_stats.items():
    c = d["count"]
    template_breakdown.append({
        "template": t,
        "samples": c,
        "anls": f"{d['anls']/c*100:.2f}%" if c > 0 else "0%",
        "exact_match": f"{d['em']/c*100:.2f}%" if c > 0 else "0%"
    })

# Xuất báo cáo JSON ra thư mục /kaggle/working
final_report = {
    "model_name": "Qwen/Qwen2-VL-2B-Instruct (Base Zero-Shot)",
    "hardware": f"Kaggle GPU {torch.cuda.get_device_name(0)}",
    "total_test_records": num_tests,
    "anls_score": round(avg_anls, 4),
    "anls_percentage": f"{avg_anls * 100:.2f}%",
    "exact_match_rate": round(avg_em, 4),
    "exact_match_percentage": f"{avg_em * 100:.2f}%",
    "avg_latency_seconds": round(avg_lat, 3),
    "vram_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024**3), 2),
    "template_breakdown": template_breakdown,
    "details": results
}

with open("/kaggle/working/baseline_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 75)
print("📊 TỔNG HỢP KẾT QUẢ BASE MODEL TRÊN 15 LOẠI HÓA ĐƠN THỰC TẾ:")
print("=" * 75)
print(f"- Tổng số mẫu kiểm định (Validation Samples) : {num_tests}")
print(f"- Điểm ANLS Score (DocVQA Metric)            : {final_report['anls_score']} ({final_report['anls_percentage']})")
print(f"- Tỉ lệ Exact Match (EM Rate)                 : {final_report['exact_match_rate']} ({final_report['exact_match_percentage']})")
print(f"- Thời gian suy luận trung bình (Avg Latency): {final_report['avg_latency_seconds']} giây / câu hỏi")
print(f"- Dung lượng VRAM tiêu thụ                   : {final_report['vram_allocated_gb']} GB")
print("=" * 75)
